In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Indlæs data
aging = pd.read_csv("age-dependency-ratio-old.csv")
pwt = pd.read_excel("Human-Productivity.xlsx", sheet_name="Data")

# Rens aging
aging = aging.rename(columns={
    "Entity": "country",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "aging"
})
aging = aging[["country", "year", "aging"]].copy()
aging["year"] = pd.to_numeric(aging["year"], errors="coerce")
aging["aging"] = pd.to_numeric(aging["aging"], errors="coerce")

# Vælg PWT-variabler
pwt = pwt[["country", "year", "rgdpo", "pop", "emp", "rkna", "hc", "rtfpna"]].copy()
pwt = pwt.rename(columns={
    "rgdpo": "Y",
    "emp": "L",
    "rkna": "K",
    "hc": "H",
    "rtfpna": "A"
})

for col in ["year", "Y", "pop", "L", "K", "H", "A"]:
    pwt[col] = pd.to_numeric(pwt[col], errors="coerce")

# Merge
df = pwt.merge(aging, on=["country", "year"], how="inner")

# Rens
df = df.dropna()
df["year"] = df["year"].astype(int)
df = df[(df["year"] >= 1990) & (df["year"] <= 2020)]

df = df[
    (df["Y"] > 0) &
    (df["pop"] > 0) &
    (df["L"] > 0) &
    (df["K"] > 0) &
    (df["H"] > 0) &
    (df["A"] > 0) &
    (df["aging"] > 0)
].copy()

# Konstruér variable
df["y_worker"] = df["Y"] / df["L"]
df["k_worker"] = df["K"] / df["L"]

df["log_y_worker"] = np.log(df["y_worker"])
df["log_k_worker"] = np.log(df["k_worker"])
df["log_H"] = np.log(df["H"])
df["log_A"] = np.log(df["A"])

# Hovedmodel
model = smf.ols(
    "log_y_worker ~ log_k_worker + log_H + log_A + aging + C(country) + C(year)",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["country"]})

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           log_y_worker   R-squared:                       0.975
Model:                            OLS   Adj. R-squared:                  0.974
Method:                 Least Squares   F-statistic:                     51.13
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           5.84e-52
Time:                        10:34:32   Log-Likelihood:                 1207.1
No. Observations:                3296   AIC:                            -2130.
Df Residuals:                    3154   BIC:                            -1264.
Df Model:                         141                                         
Covariance Type:              cluster                                         
                                             coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 141, but rank is 34
  warnings.warn('covariance of constraints does not have full '
